In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score


In [2]:

print("Loading the Parquet masterpieces...")
train_df = pd.read_parquet("../data/model_ready/train_final.parquet")
valid_df = pd.read_parquet("../data/model_ready/valid_final.parquet")
test_df = pd.read_parquet("../data/model_ready/test_final.parquet")

target_col = 'SalaryNormalized'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# THE KAGGLE CHEAT CODE (Still needed for Lasso)
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)
y_test_log = np.log1p(y_test)

scaler = StandardScaler()

# We only 'fit' the scaler on the training data so we don't cheat
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)


Loading the Parquet masterpieces...


In [3]:
# ==========================================
# TRAINING THE RUTHLESS MANAGER
# ==========================================
print("Training Lasso Regression...")
# alpha is how aggressive the manager is. 
# 0.001 is a good start. Too high and it fires every column.
lasso_model = Lasso(alpha=0.001, random_state=42, max_iter=2000) 

lasso_model.fit(X_train_scaled, y_train_log)

print("Predicting...")
y_pred_log = lasso_model.predict(X_test_scaled)

# Reverse the log
y_pred_real = np.expm1(y_pred_log)

# Metrics
mae = mean_absolute_error(y_test, y_pred_real)
r2 = r2_score(y_test, y_pred_real)

print("\n" + "="*30)
print("🤠 LASSO RESULTS 🤠")
print("="*30)
print(f"R2 Score: {r2:.4f}")
print(f"MAE:      £{mae:.2f}")
print("="*30 + "\n")

# Flex Check: Let's see how many columns Lasso actually deleted
# A coefficient of 0 means the column was fired.
fired_columns = np.sum(lasso_model.coef_ == 0)
kept_columns = np.sum(lasso_model.coef_ != 0)
print(f"Lasso kept {kept_columns} columns and completely deleted {fired_columns} useless ones.")

Training Lasso Regression...
Predicting...

🤠 LASSO RESULTS 🤠
R2 Score: 0.5066
MAE:      £5905.63

Lasso kept 204 columns and completely deleted 574 useless ones.


c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.108e+01, tolerance: 5.914e+00
  model = cd_fast.enet_coordinate_descent(
